# Timing window finder using GMM

In [5]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
import warnings

# Suppress warnings from GMM when clusters are tiny
warnings.filterwarnings('ignore')

# ==========================================
# Configuration & File Paths
# ==========================================
PATH_USER_SEGMENTS_CSV = './output/users_segmented.csv'
PATH_OUTPUT_GMM = './output/users_segmented_GMM.csv'

# ==========================================
# 1. Standard Window Mapping
# ==========================================
def map_hour_to_window(hour: float) -> str:
    """Maps a 0-23 hour float/int to the 6 predefined time windows."""
    # Handle NaN values safely
    if pd.isna(hour):
        return 'afternoon' # Safe default
        
    hour = int(round(hour)) % 24
    if 6 <= hour < 9: return 'early_morning'
    elif 9 <= hour < 12: return 'mid_morning'
    elif 12 <= hour < 15: return 'afternoon'
    elif 15 <= hour < 18: return 'late_afternoon'
    elif 18 <= hour < 21: return 'evening'
    else: return 'night'

# ==========================================
# 2. ML Optimizer Class
# ==========================================
class DynamicWindowOptimizer:
    def __init__(self, max_windows: int = 3):
        self.max_windows = max_windows
        self.segment_window_assignments = {}

    def fit(self, csv_path: str = PATH_USER_SEGMENTS_CSV):
        """Runs the ML algorithm to determine the optimal number of windows per segment."""
        # 1. Load Data from CSV
        df = pd.read_csv(csv_path)
        
        # Ensure 'preferred_hour' exists, fallback if column is named differently in your CSV
        hour_col = 'preferred_hour' if 'preferred_hour' in df.columns else 'hour'
        
        # Drop rows where the hour is missing just for the ML training
        df_clean = df.dropna(subset=[hour_col])
        
        results = []
        
        # 2. Process each segment individually
        for segment_id, group in df_clean.groupby('segment_id'):
            # Reshape for sklearn: 2D array of [[hour], [hour], ...]
            X = group[hour_col].values.reshape(-1, 1)
            
            # Edge Case: Very few users in segment, default to 1 window
            if len(X) < 5:
                mean_hour = X.mean()
                results.append({
                    'segment_id': segment_id,
                    'num_windows_assigned': 1,
                    'optimal_windows': [map_hour_to_window(mean_hour)],
                    'cluster_centers': [round(mean_hour, 1)]
                })
                self.segment_window_assignments[segment_id] = [map_hour_to_window(mean_hour)]
                continue

            # 3. Fit GMMs for k=1, 2, and 3 to find the optimal number of clusters
            best_gmm = None
            lowest_bic = np.inf
            
            # Test 1, 2, and 3 distinct time clusters
            max_k_to_try = min(self.max_windows, len(np.unique(X))) 
            
            for k in range(1, max_k_to_try + 1):
                gmm = GaussianMixture(n_components=k, random_state=42)
                gmm.fit(X)
                bic_score = gmm.bic(X)
                
                # If this model fits the data better (lower BIC), save it
                if bic_score < lowest_bic:
                    lowest_bic = bic_score
                    best_gmm = gmm
            
            # 4. Extract the optimal hours (cluster centers) from the best model
            optimal_hours = best_gmm.means_.flatten()
            
            # Map those hours back to your 6 standard text windows
            optimal_windows = list(set([map_hour_to_window(h) for h in optimal_hours]))
            
            results.append({
                'segment_id': segment_id,
                'num_windows_assigned': len(optimal_windows),
                'optimal_windows': optimal_windows,
                'cluster_centers': [round(h, 1) for h in optimal_hours]
            })
            
            # Save to internal state
            self.segment_window_assignments[segment_id] = optimal_windows

        return pd.DataFrame(results)

# ==========================================
# 3. Execution & Export
# ==========================================
if __name__ == "__main__":
    ml_optimizer = DynamicWindowOptimizer(max_windows=3)
    
    try:
        print(f"Loading data from {PATH_USER_SEGMENTS_CSV}...")
        df_optimal = ml_optimizer.fit(PATH_USER_SEGMENTS_CSV)
        
        print("\nML-Assigned Optimal Time Windows per Segment:")
        print(df_optimal.to_string(index=False))
        
        # Export the final optimal windows to the output directory
        df_optimal.to_csv(PATH_OUTPUT_GMM, index=False)
        print(f"\nSuccessfully exported GMM mappings to: {PATH_OUTPUT_GMM}")
        
    except FileNotFoundError:
        print(f"Error: Could not find '{PATH_USER_SEGMENTS_CSV}'. Check the path.")
    except KeyError as e:
        print(f"Error: Column {e} not found in the CSV. Please verify column names.")

Loading data from ./output/users_segmented.csv...

ML-Assigned Optimal Time Windows per Segment:
segment_id  num_windows_assigned                        optimal_windows   cluster_centers
        S1                     3       [evening, night, late_afternoon] [20.0, 5.0, 17.0]
        S2                     3          [mid_morning, evening, night] [8.7, 19.0, 22.0]
        S3                     1                       [late_afternoon]            [16.0]
        S4                     2               [early_morning, evening] [18.0, 7.0, 19.0]
        S5                     3    [early_morning, afternoon, evening] [7.0, 20.2, 13.0]
        S6                     3 [mid_morning, evening, late_afternoon] [19.6, 9.3, 15.0]
        S7                     2                   [mid_morning, night]       [20.5, 9.1]
        S8                     1                                [night]            [21.0]
        S9                     1                              [evening]            [19.2]

Su